# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ikramkhan-gif1/FlyRank-ML-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Setup: Install Libraries and Connect to Data Warehouse

In [1]:
pip install datasets duckdb

In [4]:
import duckdb
import pandas as pd
import os
from huggingface_hub import list_repo_files, hf_hub_download

# Define the Hugging Face repository and the specific subfolder for March 2026 data
repo_id = 'FlyRank/internship-warehouse'
subfolder = 'fact_content_daily_performance/month=2026-03/'
local_dir = '/tmp/march_2026_data'

# Create local directory if it doesn't exist
os.makedirs(local_dir, exist_ok=True)

print(f"Downloading March 2026 data from '{repo_id}/{subfolder}' to '{local_dir}'...")

# List all files in the repository and filter for March 2026 parquet files
# Specify repo_type='dataset' to correctly target the Hugging Face dataset
all_files_in_repo = list_repo_files(repo_id=repo_id, repo_type='dataset')
march_parquet_files = [f for f in all_files_in_repo if f.startswith(subfolder) and f.endswith('.parquet')]

if not march_parquet_files:
    raise FileNotFoundError(f"No parquet files found in '{subfolder}' for March 2026.")

# Download each March 2026 parquet file
local_parquet_paths = []
for file_path in march_parquet_files:
    local_path = hf_hub_download(
        repo_id=repo_id,
        filename=file_path,
        repo_type='dataset', # Specify repo_type for download as well
        local_dir=local_dir,
        local_dir_use_symlinks=False # Ensure actual files are downloaded
    )
    local_parquet_paths.append(local_path)

print(f"Successfully downloaded {len(local_parquet_paths)} March 2026 parquet files.")

# Initialize DuckDB connection
con = duckdb.connect(database=':memory:')

# Register the local March 2026 parquet files as a DuckDB view
# The glob pattern '*.parquet' will read all downloaded parquet files from the correct subfolder
con.execute(f"""
    CREATE OR REPLACE VIEW fact_content_daily_performance AS
    SELECT * FROM '{local_dir}/{subfolder}*.parquet';
""")

print("DuckDB connected and 'fact_content_daily_performance' view registered for local March 2026 data.")


Successfully downloaded 1 March 2026 parquet files.
DuckDB connected and 'fact_content_daily_performance' view registered for local March 2026 data.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


In [8]:
import pandas as pd # Ensure pandas is imported for fetchdf() in case it wasn't already

# Inspect the actual schema as the single source of truth
query_schema = "PRAGMA table_info('fact_content_daily_performance');"
schema_df = con.execute(query_schema).fetchdf()
display(schema_df)

,cid,name,type,notnull,dflt_value,pk
0,0,report_date,DATE,False,None,False
1,1,client_hash_id,VARCHAR,False,None,False
2,2,content_hash_id,VARCHAR,False,None,False
3,3,client_has_gsc,BOOLEAN,False,None,False
4,4,client_has_ga4,BOOLEAN,False,None,False
5,5,gsc_data_available,BOOLEAN,False,None,False
6,6,ga4_data_available,BOOLEAN,False,None,False
7,7,gsc_impressions,BIGINT,False,None,False
8,8,gsc_clicks,BIGINT,False,None,False
9,9,gsc_sum_position,BIGINT,False,None,False


In [7]:
# Verify the unit of analysis and time window for March 2026
query = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT (report_date, client_hash_id, content_hash_id)) AS distinct_grain_rows,
    MIN(report_date) AS min_report_date,
    MAX(report_date) AS max_report_date
FROM fact_content_daily_performance
WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31';
"""
display(con.execute(query).fetchdf())

query_duplicates = """
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS duplicate_count
FROM fact_content_daily_performance
WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
GROUP BY report_date, client_hash_id, content_hash_id
HAVING COUNT(*) > 1;
"""
display(con.execute(query_duplicates).fetchdf())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

RuntimeError: Query interrupted

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Unit of Analysis
One row = one unique combination of `report_date`, `client_hash_id`, and `content_hash_id`.

### Time Window
The development time window is **March 2026** (i.e., `report_date` from '2026-03-01' to '2026-03-31', inclusive).

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [9]:
# Verify the unit of analysis and time window for March 2026
# This cell was originally intended for grain verification, but is now superseded by 6b63acae
# Kept for completeness of modification based on user instructions to correct all occurrences.
query = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT (report_date, client_hash_id, content_hash_id)) AS distinct_grain_rows,
    MIN(report_date) AS min_report_date,
    MAX(report_date) AS max_report_date
FROM fact_content_daily_performance
WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31';
"""
display(con.execute(query).fetchdf())

query_duplicates = """
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS duplicate_count
FROM fact_content_daily_performance
WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
GROUP BY report_date, client_hash_id, content_hash_id
HAVING COUNT(*) > 1;
"""
display(con.execute(query_duplicates).fetchdf())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,distinct_grain_rows,min_report_date,max_report_date
0,9841378,9841378,2026-03-01,2026-03-31


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,duplicate_count


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Schema Inspection
First, let's inspect the schema of the `fact_content_daily_performance` table to understand available columns.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [10]:
# Inspect the schema of the fact_content_daily_performance table
# This cell is redundant as schema was already inspected in cell 9b18479f
query = "PRAGMA table_info('fact_content_daily_performance');"
display(con.execute(query).fetchdf())

,cid,name,type,notnull,dflt_value,pk
0,0,report_date,DATE,False,None,False
1,1,client_hash_id,VARCHAR,False,None,False
2,2,content_hash_id,VARCHAR,False,None,False
3,3,client_has_gsc,BOOLEAN,False,None,False
4,4,client_has_ga4,BOOLEAN,False,None,False
5,5,gsc_data_available,BOOLEAN,False,None,False
6,6,ga4_data_available,BOOLEAN,False,None,False
7,7,gsc_impressions,BIGINT,False,None,False
8,8,gsc_clicks,BIGINT,False,None,False
9,9,gsc_sum_position,BIGINT,False,None,False


### Categorization of Fields
Based on the schema inspection and common patterns for search intelligence data, here's a proposed categorization:

*   **Label/Proxy:**
    *   `gsc_impressions`: Represents the total number of times content was displayed from Google Search Console. This is a direct measure of performance.

*   **Features (Max 5):**
    *   `gsc_clicks`: Total number of clicks from Google Search Console. (Could be considered a feature depending on the task, or a more direct label if predicting click-through-rate).
    *   `gsc_avg_position`: Average search result position from Google Search Console. Lower is generally better.
    *   `ga4_pageviews`: Total number of pageviews from GA4. Useful for understanding engagement.
    *   `sessions_organic`: Number of organic sessions. Indicates traffic from organic search.
    *   `sessions_social`: Number of social media sessions. Indicates traffic from social platforms.

*   **Context:**
    *   `report_date`: The date of the performance record.
    *   `client_hash_id`: Hashed identifier for the client.
    *   `content_hash_id`: Hashed identifier for the content item.
    *   `client_has_gsc`: Boolean indicating if client has GSC data.
    *   `client_has_ga4`: Boolean indicating if client has GA4 data.
    *   `gsc_data_available`: Boolean indicating if GSC data is available for this row.
    *   `ga4_data_available`: Boolean indicating if GA4 data is available for this row.
    *   `month`: The month of the record.

*   **Excluded:**
    *   `cid`: Internal column ID from PRAGMA, not part of the actual data. Provides no predictive power.
    *   `name`: Column name from PRAGMA, not part of the actual data. Provides no predictive power.
    *   `type`: Column type from PRAGMA, not part of the actual data. Provides no predictive power.
    *   `notnull`: Not-null constraint from PRAGMA, not part of the actual data. Provides no predictive power.
    *   `dflt_value`: Default value from PRAGMA, not part of the actual data. Provides no predictive power.
    *   `pk`: Primary key indicator from PRAGMA, not part of the actual data. Provides no predictive power.
    *   `gsc_sum_position`: Sum of positions, typically `gsc_avg_position` is more directly useful for modeling.
    *   `ga4_users`: Number of users from GA4. Highly correlated with pageviews/sessions, can cause multicollinearity.
    *   `ga4_engaged_sessions`: Number of engaged sessions from GA4. Highly correlated with total sessions.
    *   `ga4_total_engagement_sec`: Total engagement time from GA4. Can be a noisy signal and highly correlated with sessions.
    *   `sessions_direct`: Direct sessions. Might not be as informative for *search intelligence* as organic/social.
    *   `sessions_referral`: Referral sessions. Could be noisy depending on referral sources.
    *   `sessions_paid`: Paid sessions. Not directly relevant to organic search intelligence in this context.
    *   `sessions_ai`, `ai_chatgpt`, `ai_perplexity`, `ai_gemini`, `ai_copilot`, `ai_claude`, `ai_meta`, `ai_other`: AI-related metrics. While interesting, for a basic search intelligence model, these might introduce complexity or noise without clear immediate value, and are often very sparse.
    *   `scroll_events`: Number of scroll events. Engagement metric, but `ga4_pageviews` and `gsc_clicks` might be stronger primary signals.
    *   `target_variable_for_leakage_demo`: (Will be a synthetic column for demonstration purposes only, designed to show leakage and then explicitly excluded from the final feature set).

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### 3.1 Grain Verification
Verifying that each row represents a unique `report_date` × `client` × `content` combination for March 2026.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [11]:
# Check for duplicate grain entries for March 2026
query = """
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS num_occurrences
FROM fact_content_daily_performance
WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
GROUP BY report_date, client_hash_id, content_hash_id
HAVING COUNT(*) > 1;
"""
duplicate_grain = con.execute(query).fetchdf()

if duplicate_grain.empty:
    print("No duplicate grain entries found for March 2026.")
else:
    print("Duplicate grain entries found:")
    display(duplicate_grain)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

No duplicate grain entries found for March 2026.


In [14]:
# March 2026 row count and date span
query = """
SELECT
    COUNT(*) AS total_rows_march_2026,
    MIN(report_date) AS min_date_march_2026,
    MAX(report_date) AS max_date_march_2026
FROM fact_content_daily_performance
WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31';
"""
display(con.execute(query).fetchdf())

,total_rows_march_2026,min_date_march_2026,max_date_march_2026
0,9841378,2026-03-01,2026-03-31


### 3.2 March 2026 Row Count and Date Span
Confirmed the total number of rows (9,841,378) and the exact date range (2026-03-01 to 2026-03-31) for the March 2026 development window.

In [15]:
# This cell is now redundant as its purpose was fulfilled by 9f82cced, which executed successfully.
# Removing to clean up the notebook and avoid previous NameError.


### 3.3 Availability Check (`IS NOT NULL`)
Checking the availability of key label/feature columns using `IS NOT NULL` to ensure data quality. We are checking `gsc_impressions` (label proxy), `gsc_clicks`, and `gsc_avg_position`.

In [12]:
# Availability using IS NOT NULL for key columns
query = """
SELECT
    COUNT(*) AS total_march_rows,
    COUNT(gsc_impressions) AS gsc_impressions_available,
    COUNT(gsc_clicks) AS gsc_clicks_available,
    COUNT(gsc_avg_position) AS gsc_avg_position_available
FROM fact_content_daily_performance
WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31';
"""
display(con.execute(query).fetchdf())

,total_march_rows,gsc_impressions_available,gsc_clicks_available,gsc_avg_position_available
0,9841378,9841378,9841378,3611061


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### 4. Data Limits

This dataset for March 2026 has several inherent limitations:

*   **Limited Time Window:** It only covers a single month (March 2026). This short duration makes it difficult to detect long-term trends, seasonality spanning multiple years, or gradual changes in content performance. It might not be representative of other periods, especially if there are significant external events.
*   **No Future Data:** The dataset explicitly excludes future data (June 2026 is sealed). This means we cannot use any information from the outcome month for feature engineering, preventing look-ahead bias.
*   **Specific Grain:** The grain (`report_date` × `client` × `content`) means we cannot analyze overall client performance across all content, or overall content performance across all clients directly without aggregation. Aggregations would create a different unit of analysis.
*   **Potential for Unbalanced History:** Without data from prior periods, we cannot assess if the historical performance leading into March 2026 was typical or an anomaly.
*   **Source Limitations:** If this data originates primarily from a specific source (e.g., Google Search Console), it might not capture performance from other search engines, direct traffic, or other referral sources, leading to an incomplete picture of content reach and impact.
*   **Feature Completeness:** The available features might not capture all relevant aspects influencing content performance (e.g., content quality, external marketing campaigns, competitor activity). Missing these factors can lead to an incomplete understanding and less predictive models.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Leakage Demonstration

To demonstrate feature leakage, we will simulate a scenario where a feature is directly derived from the label (`impressions_sum`). We will create a synthetic `target_variable_for_leakage_demo` feature, which is `impressions_sum` plus some noise. We will then show how this leaked feature significantly boosts a simple model's performance, making the model misleadingly accurate. Finally, we will explicitly exclude this feature from the final feature set.

In [13]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# Load a sample of data for March 2026
df_march = con.execute("""
SELECT
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position
FROM fact_content_daily_performance
WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
AND gsc_impressions IS NOT NULL
AND gsc_clicks IS NOT NULL
AND gsc_avg_position IS NOT NULL;
""").fetchdf()

# Drop rows with any NaN values for this demo
df_march = df_march.dropna()

# Define the label
y = df_march['gsc_impressions']

# Create a synthetic leaked feature
# This feature is directly derived from the label, simulating leakage
df_march['target_variable_for_leakage_demo'] = y * 1.1 + np.random.normal(0, 0.05 * y.std(), len(y))

# Case 1: Model with leaked feature
X_leaked = df_march[['gsc_clicks', 'gsc_avg_position', 'target_variable_for_leakage_demo']]
X_train_leaked, X_test_leaked, y_train_leaked, y_test_leaked = train_test_split(X_leaked, y, test_size=0.2, random_state=42)

model_leaked = LinearRegression()
model_leaked.fit(X_train_leaked, y_train_leaked)
y_pred_leaked = model_leaked.predict(X_test_leaked)

rmse_leaked = np.sqrt(mean_squared_error(y_test_leaked, y_pred_leaked))
r2_leaked = r2_score(y_test_leaked, y_pred_leaked)

print("--- Model with Leaked Feature ---")
print(f"RMSE: {rmse_leaked:.2f}")
print(f"R-squared: {r2_leaked:.2f}")

# Case 2: Model without leaked feature (proper feature set)
X_clean = df_march[['gsc_clicks', 'gsc_avg_position']]
X_train_clean, X_test_clean, y_train_clean, y_test_clean = train_test_split(X_clean, y, test_size=0.2, random_state=42)

model_clean = LinearRegression()
model_clean.fit(X_train_clean, y_train_clean)
y_pred_clean = model_clean.predict(X_test_clean)

rmse_clean = np.sqrt(mean_squared_error(y_test_clean, y_pred_clean))
r2_clean = r2_score(y_test_clean, y_pred_clean)

print("\n--- Model without Leaked Feature (Clean) ---")
print(f"RMSE: {rmse_clean:.2f}")
print(f"R-squared: {r2_clean:.2f}")

print("\nObservation: The model with the 'target_variable_for_leakage_demo' feature shows significantly higher R-squared and lower RMSE, indicating leakage. This feature will be excluded from the final feature set.")

# Explicitly remove the leaked column from the intended final feature set
final_features = ['gsc_clicks', 'gsc_avg_position', 'ga4_pageviews', 'sessions_organic', 'sessions_social'] # Assuming these are the selected features after leakage check
print(f"\nFinal features for the model (excluding leaked feature): {final_features}")

--- Model with Leaked Feature ---
RMSE: 11.34
R-squared: 1.00

--- Model without Leaked Feature (Clean) ---
RMSE: 202.79
R-squared: 0.35

Observation: The model with the 'target_variable_for_leakage_demo' feature shows significantly higher R-squared and lower RMSE, indicating leakage. This feature will be excluded from the final feature set.

Final features for the model (excluding leaked feature): ['gsc_clicks', 'gsc_avg_position', 'ga4_pageviews', 'sessions_organic', 'sessions_social']


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.